# 🏠 Argenprop Scrapper
Scrapper de departamentos en venta en Capital Federal usando [argenprop.com](https://www.argenprop.com).

> **Fuente:** https://github.com/nachols1986/argenprop_scrapper

## 1. Instalación de dependencias

In [ ]:
%pip install requests beautifulsoup4 pandas

Note: you may need to restart the kernel to use updated packages.


## 2. Imports y funciones auxiliares

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import os

def clean_text(text):
    if not text: return "N/A"
    text = text.replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def parse_address(address_raw):
    calle = altura = piso = "N/A"
    try:
        address_raw = address_raw.replace("Piso ", "").replace("piso ", "")
        match = re.search(r'^(.*?)\s(\d+)(?:,\s?(.*))?$', address_raw)
        if match:
            calle = match.group(1).strip()
            altura = match.group(2).strip()
            piso = match.group(3).strip() if match.group(3) else "0"
        else:
            calle = address_raw
    except: pass
    return calle, altura, piso

def get_description(url, headers):
    try:
        res = requests.get(url, headers=headers, timeout=10)
        if res.status_code == 200:
            s = BeautifulSoup(res.content, 'html.parser')
            desc = s.find('section', class_='section-description')
            if desc:
                return clean_text(desc.text).replace("Leer más Leer menos", "").strip()
    except: pass
    return "Sin descripción"

def extract_smart_features(row):
    texto = (str(row['Descripción']) + " " + str(row['Detalles'])).lower()
    return pd.Series({
        "Amenities": 1 if any(x in texto for x in ["amenities", "piscina", "pileta", "sum", "parrilla", "gym", "sauna", "laundry"]) else 0,
        "Losa_Central": 1 if any(x in texto for x in ["losa radiante", "calefacción central", "caldera central", "piso radiante"]) else 0,
        "Aire_Acond": 1 if any(x in texto for x in ["aire acondicionado", "split", " a/c", "frío-calor"]) else 0,
        "Apto_Credito": 1 if "apto crédito" in texto or "apto credito" in texto else 0,
        "Cochera": 1 if any(x in texto for x in ["cochera", "espacio guarda coche", "estacionamiento", "guarda coche"]) else 0,
        "Seguridad": 1 if any(x in texto for x in ["vigilancia", "seguridad 24", "tótem", "totem", "encargado"]) else 0,
        "Luminoso": 1 if any(x in texto for x in ["luminoso", "todo luz", "vista abierta", "vista panorámica", "sol"]) else 0,
        "Balcon_Aterrazado": 1 if "aterrazado" in texto or "balcón terraza" in texto else 0
    })

print("✅ Funciones cargadas correctamente.")

✅ Funciones cargadas correctamente.


## 3. Función principal del scrapper

In [3]:
def run_scrapper(max_pages=3):
    base_url = "https://www.argenprop.com/departamentos/venta/capital-federal"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"}
    
    all_data = []
    seen_links = set()
    output_dir = "output"
    if not os.path.exists(output_dir): os.makedirs(output_dir)

    for i in range(1, max_pages + 1):
        url = f"{base_url}?pagina-{i}" if i > 1 else base_url
        print(f"\n--- PROCESANDO PÁGINA {i} ---")
        print(f"URL: {url}")
        
        try:
            r = requests.get(url, headers=headers)
            soup = BeautifulSoup(r.content, 'html.parser')
            items = soup.find_all('div', class_='listing__item')

            if not items:
                print("⚠️ No se encontraron items en esta página. Deteniendo.")
                break

            for item in items:
                try:
                    link_tag = item.find('a', class_='card')
                    if not link_tag: continue
                    link = "https://www.argenprop.com" + link_tag['href']
                    
                    if link in seen_links: continue
                    seen_links.add(link)

                    price_block = item.find('p', class_='card__price')
                    price_text = clean_text(price_block.text) if price_block else ""
                    p_match = re.search(r'(USD|S)\s?([\d.]+)', price_text)
                    precio = p_match.group(0) if p_match else "Consultar"
                    e_match = re.search(r'\+\s?\$?\s?([\d.]+)', price_text)
                    expensas = e_match.group(0) if e_match else "N/A"

                    raw_address = clean_text(item.find('p', class_='card__address').text)
                    calle, altura, piso = parse_address(raw_address)
                    features = clean_text(item.find('ul', class_='card__main-features').text)
                    
                    print(f"  -> Extrayendo: {calle} {altura}...")
                    desc = get_description(link, headers)
                    
                    all_data.append({
                        "Precio": precio, "Expensas": expensas, "Calle": calle,
                        "Altura": altura, "Piso": piso, "Detalles": features,
                        "Descripción": desc, "Link": link
                    })
                    time.sleep(1.2)
                except: continue
        except Exception as e:
            print(f"❌ Error en página {i}: {e}")
            break

    if all_data:
        df = pd.DataFrame(all_data)
        features_df = df.apply(extract_smart_features, axis=1)
        df = pd.concat([df, features_df], axis=1)
        
        filename = f"argenprop_export_{int(time.time())}.tsv"
        filepath = os.path.join(output_dir, filename)
        df.to_csv(filepath, sep='\t', index=False, encoding='utf-8-sig')
        print(f"\n✅ ¡Éxito! Archivo guardado en: {filepath}")
        return df
    else:
        print("⚠️ No se obtuvieron datos.")
        return None

print("✅ Función run_scrapper() lista.")

✅ Función run_scrapper() lista.


## 4. ▶️ Ejecutar el scrapper

Podés cambiar `max_pages` para scrapear más o menos páginas. Cada página tiene ~20 propiedades.

In [4]:
df = run_scrapper(max_pages=3)


--- PROCESANDO PÁGINA 1 ---
URL: https://www.argenprop.com/departamentos/venta/capital-federal
  -> Extrayendo: Castex 3300...
  -> Extrayendo: GURRUCHAGA 2100...
  -> Extrayendo: Jerónimo Salguero 2700...
  -> Extrayendo: Bulnes 1600...
  -> Extrayendo: Av. Gral. Indalecio Chenaut 1900...
  -> Extrayendo: BORGES JORGE LUIS 2100...
  -> Extrayendo: Soler 5800...
  -> Extrayendo: LERMA 300...
  -> Extrayendo: JULIAN ALVAREZ 1600...
  -> Extrayendo: JOSE ANTONIO CABRERA 3600...
  -> Extrayendo: Jerónimo Salguero 1900...
  -> Extrayendo: Gurruchaga 1100...
  -> Extrayendo: AV FRANCISCO ACUÑA DE FIGUEROA 1200...
  -> Extrayendo: Santa Rosa 5000...
  -> Extrayendo: Bulnes 1300...
  -> Extrayendo: Honduras 3900...
  -> Extrayendo: ZAPATA 300...
  -> Extrayendo: RIVADAVIA, AV. 5200...
  -> Extrayendo: CONESA 2500...
  -> Extrayendo: AMENABAR 2100...

--- PROCESANDO PÁGINA 2 ---
URL: https://www.argenprop.com/departamentos/venta/capital-federal?pagina-2
  -> Extrayendo: Balbin 2400...
  -> Ex

## 5. Vista previa de los resultados

In [5]:
if df is not None:
    print(f"Total de propiedades scrapeadas: {len(df)}")
    df.head(10)

Total de propiedades scrapeadas: 60


## 6. Análisis rápido (opcional)

In [6]:
if df is not None:
    cols_features = ["Amenities", "Losa_Central", "Aire_Acond", "Apto_Credito",
                     "Cochera", "Seguridad", "Luminoso", "Balcon_Aterrazado"]
    print("📊 Porcentaje de propiedades con cada característica:")
    print((df[cols_features].mean() * 100).round(1).astype(str) + "%")

📊 Porcentaje de propiedades con cada característica:
Amenities            41.7%
Losa_Central         11.7%
Aire_Acond           45.0%
Apto_Credito         15.0%
Cochera              31.7%
Seguridad            30.0%
Luminoso             80.0%
Balcon_Aterrazado    13.3%
dtype: object


In [7]:
df.head()

,Precio,Expensas,Calle,Altura,Piso,Detalles,Descripción,Link,Amenities,Losa_Central,Aire_Acond,Apto_Credito,Cochera,Seguridad,Luminoso,Balcon_Aterrazado
0,USD 570.000,+ $1.000.000,Castex,3300,0,140 m² cubie. 3 dorm. 40 años,Piso alto con vista abierta en torre Arq Clori...,https://www.argenprop.com/departamento-en-vent...,0,1,0,0,1,1,1,1
1,USD 98.000,+ $150.000,GURRUCHAGA,2100,5,31 m² cubie. 15 años 1 baño,"Departamento - Venta - Argentina, Capital Fede...",https://www.argenprop.com/departamento-en-vent...,0,1,0,0,0,0,1,0
2,USD 2.500.000,+ $2.400.000,Jerónimo Salguero,2700,0,275 m² cubie. 3 dorm. 17 años,TORRE BELLINI! Piso alto de revista! Balcón co...,https://www.argenprop.com/departamento-en-vent...,1,0,1,1,1,1,1,0
3,USD 155.000,+ $260.000,Bulnes,1600,0,60 m² cubie. 2 dorm. 30 años,IMPECABLE! BALCÓN CORRIDO AL FRENTE CON VISTA ...,https://www.argenprop.com/departamento-en-vent...,0,0,1,1,1,1,1,0
4,USD 181.000,+ $335.000,Av. Gral. Indalecio Chenaut,1900,0,75 m² cubie. 1 dorm. 20 años,IMPECABLE DÚPLEX! TODO A NUEVO! Balcón con esp...,https://www.argenprop.com/departamento-en-vent...,1,0,1,0,0,1,1,0
